In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    datediff,
    from_utc_timestamp,
    lit,
    sha2,
    when,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_payment_schedule_process(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "PaymentScheduleId string, SourceSystem string, PortfolioKey String, PortfolioName string,"
        #     "IngestionTimestamp timestamp,  BatchId string, PaymentScheduleKey string,"
        #     "ExposureKey String, PartyKey string, AgreementKey string, SettlementKey String, PaymentScheduleTypeCode string, PaymentScheduleType string,"
        #     "InstallmentNumber int, InstallmentDate timestamp, RequestSN int, InstallmentTypeCode string,"
        #     "InstallmentType string, InstallmentStatusCode string, InstallmentStatus string,"
        #     "InstallmentAmount decimal(38,7), InterestAmount decimal(19,2), CapitalAmount decimal(19,2),"
        #     "CommissionAmount decimal(19,2), ExpenseAmount decimal(19,2),"
        #     "OverdueDays int, InstallmentAmountPaid decimal(19,2), "
        #     "InterestAmountPaid decimal(19,2), CapitalAmountPaid decimal(19,2),"
        #     "CommissionAmountPaid decimal(19,2), ExpenseAmountPaid decimal(19,2),"
        #     "RemainingAmount decimal(19,2), DownPaymentFlag boolean, RecordKey int,"
        #     "SettlementCaseRecordKey int, PayTolerance int, PayPercent decimal(12,6), PayPercentAmount decimal(18,2),"
        #     "ImplementationFlag boolean ,BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.PaymentSchedule"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "process",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== Portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter

        portfolio_selected_fields = ["PortfolioName", "vendor_code_qualco"]

        portfolio = load_landed_portfolio_table(
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # ================== settlement_plan ==================

        settlement_plan_extra_filters = None

        if settlement_plan_extra_filters is not None:
            settlement_plan_extra_filters.append(date_filter)
            settlement_plan_final_filters = " AND ".join(settlement_plan_extra_filters)
        else:
            settlement_plan_final_filters = date_filter

        settlement_plan_selected_fields = [
            "settlement_code",
            "install_no",
            "install_date",
            "fulfilled_flg",
            "install_amount",
            "paid_amn",
            "down_payment_flg",
            "businessdate",
        ]

        settlement_plan = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "process_settlement_plan",
            settlement_plan_final_filters,
            settlement_plan_selected_fields,
        )

        # ================== settlement ==================

        settlement_extra_filters = None

        if settlement_extra_filters is not None:
            settlement_extra_filters.append(date_filter)
            settlement_final_filters = " AND ".join(settlement_extra_filters)
        else:
            settlement_final_filters = date_filter

        settlement_selected_fields = [
            "vendor_code",
            "settlement_code",
            "implementation_dt",
            "businessdate",
        ]

        settlement = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "process_settlement",
            settlement_final_filters,
            settlement_selected_fields,
        )

        # Joins
        settlement_settlement_plan_RestructureType_join_condition = (
            settlement["settlement_code"] == settlement_plan["settlement_code"]
        )

        settlement_portfolio_join_condition = (
            settlement["vendor_code"] == portfolio["vendor_code_qualco"]
        )

        target_table_process = settlement.join(
            settlement_plan,
            settlement_settlement_plan_RestructureType_join_condition,
            "inner",
        ).join(portfolio, settlement_portfolio_join_condition, "left")

        # HashKeys
        target_table_process_PaymentScheduleId_primary_key_fields = [
            settlement_plan.settlement_code,
            settlement_plan.install_no,
            lit("process"),
        ]

        target_table_process = (
            target_table_process.withColumn(
                "PaymentScheduleId",
                sha2(
                    concat_ws(
                        ";", *target_table_process_PaymentScheduleId_primary_key_fields
                    ),
                    256,
                ),
            )
            .withColumn(
                "PaymentScheduleKey",
                concat_ws(
                    ";",
                    settlement_plan.settlement_code,
                    settlement_plan.install_no,
                ),
            )
            .withColumn(
                "InstallmentStatusCode",
                when(
                    settlement_plan.install_date > settlement_plan.businessdate,
                    "Active",
                )
                .when(
                    (settlement_plan.install_date <= settlement_plan.businessdate)
                    & (settlement_plan.fulfilled_flg == 1),
                    "Inactive",
                )
                .otherwise("Overdue"),
            )
            .withColumn(
                "OverdueDays",
                when(
                    (settlement_plan.install_date <= settlement_plan.businessdate)
                    & (settlement_plan.fulfilled_flg != 1),
                    datediff(
                        settlement_plan.install_date, settlement_plan.businessdate
                    ),
                ).otherwise(0),
            )
            .withColumn(
                "RemainingAmount",
                settlement_plan.install_amount - settlement_plan.paid_amn,
            ).withColumn(
                "ImplementationFlag",
                when(settlement.implementation_dt.isNotNull(), True).otherwise(False)
            )
        )
        # ================== Selection of Final Columns ==================

        result_df_process = target_table_process.select(
            target_table_process.PaymentScheduleId.alias("PaymentScheduleId"),
            lit("process").cast("String").alias("SourceSystem"),
            portfolio.vendor_code_qualco.cast("String").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("String").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit(None).cast("String").alias("BatchId"),
            target_table_process.PaymentScheduleKey.cast("String").alias(
                "PaymentScheduleKey"
            ),
            lit(None).cast("String").alias("ExposureKey"),
            lit(None).cast("String").alias("PartyKey"),
            lit(None).cast("String").alias("AgreementKey"),
            settlement_plan.settlement_code.cast("String").alias("SettlementKey"),
            lit("Settlement Payment Schedule")
            .cast("String")
            .alias("PaymentScheduleTypeCode"),
            lit("Settlement Payment Schedule")
            .cast("String")
            .alias("PaymentScheduleType"),
            settlement_plan.install_no.cast("Int").alias("InstallmentNumber"),
            settlement_plan.install_date.cast("timestamp").alias("InstallmentDate"),
            lit(None).cast("Int").alias("RequestSN"),
            lit("installment").cast("String").alias("InstallmentTypeCode"),
            lit("installment").cast("String").alias("InstallmentType"),
            target_table_process.InstallmentStatusCode.cast("String").alias(
                "InstallmentStatusCode"
            ),
            target_table_process.InstallmentStatusCode.cast("String").alias(
            "InstallmentStatus"
            ),
            settlement_plan.install_amount.cast("decimal(38,7)").alias(
                "InstallmentAmount"
            ),
            lit(None).cast("decimal(19,2)").alias("InterestAmount"),
            lit(None).cast("decimal(19,2)").alias("CapitalAmount"),
            lit(None).cast("decimal(19,2)").alias("CommissionAmount"),
            lit(None).cast("decimal(19,2)").alias("ExpenseAmount"),
            target_table_process.OverdueDays.cast("Int").alias("OverdueDays"),
            settlement_plan.paid_amn.cast("decimal(19,2)").alias(
                "InstallmentAmountPaid"
            ),
            lit(None).cast("decimal(19,2)").alias("InterestAmountPaid"),
            lit(None).cast("decimal(19,2)").alias("CapitalAmountPaid"),
            lit(None).cast("decimal(19,2)").alias("CommissionAmountPaid"),
            lit(None).cast("decimal(19,2)").alias("ExpenseAmountPaid"),
            target_table_process.RemainingAmount.cast("decimal(19,2)").alias(
                "RemainingAmount"
            ),
            settlement_plan.down_payment_flg.cast("boolean").alias("DownPaymentFlag"),
            lit(None).cast("int").alias("RecordKey"),
            lit(None).cast("int").alias("SettlementCaseRecordKey"),
            lit(None).cast("int").alias("PayTolerance"),
            lit(None).cast("decimal(12,6)").alias("PayPercent"),
            lit(None).cast("decimal(18,2)").alias("PayPercentAmount"),
            target_table_process.ImplementationFlag.cast("boolean").alias("ImplementationFlag"),
            settlement_plan.businessdate.cast("date").alias("BusinessDate"),
        )

        result_df_process.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_payment_schedule_process(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)